In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [55]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam

In [64]:
BASE_DIR = "/content/drive/MyDrive/eras_data_split"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 5


In [69]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 3052 images belonging to 5 classes.
Found 652 images belonging to 5 classes.
Found 659 images belonging to 5 classes.


In [70]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # Stage 1: freeze base

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_10     │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [71]:
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint("era_best_model.h5", monitor='val_accuracy', save_best_only=True)


In [72]:
history_stage1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 539ms/step - accuracy: 0.4105 - loss: 1.5089

96/96 ━━━━━━━━━━━━━━━━━━━━ 74s 670ms/step - accuracy: 0.4112 - loss: 1.5071 - val_accuracy: 0.4816 - val_loss: 1.2034
Epoch 2/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step - accuracy: 0.5697 - loss: 1.0929

96/96 ━━━━━━━━━━━━━━━━━━━━ 51s 536ms/step - accuracy: 0.5698 - loss: 1.0926 - val_accuracy: 0.5322 - val_loss: 1.1479
Epoch 3/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 489ms/step - accuracy: 0.6172 - loss: 0.9760

96/96 ━━━━━━━━━━━━━━━━━━━━ 51s 527ms/step - accuracy: 0.6172 - loss: 0.9758 - val_accuracy: 0.5445 - val_loss: 1.1299
Epoch 4/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 500ms/step - accuracy: 0.6557 - loss: 0.8851

96/96 ━━━━━━━━━━━━━━━━━━━━ 52s 537ms/step - accuracy: 0.6558 - loss: 0.8852 - val_accuracy: 0.5506 - val_loss: 1.1050
Epoch 5/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 496ms/step - accuracy: 0.6680 - loss: 0.8404

96/96 ━━━━━━━━━━━━━━━━━━━━ 51s 532ms/step - accuracy: 0.6679 - loss: 0.8405 - val_accuracy: 0.5552 - val_loss: 1.0599
Epoch 6/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 500ms/step - accuracy: 0.6895 - loss: 0.8005

96/96 ━━━━━━━━━━━━━━━━━━━━ 52s 537ms/step - accuracy: 0.6895 - loss: 0.8005 - val_accuracy: 0.5629 - val_loss: 1.0850
Epoch 7/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 495ms/step - accuracy: 0.7036 - loss: 0.7372

96/96 ━━━━━━━━━━━━━━━━━━━━ 51s 534ms/step - accuracy: 0.7035 - loss: 0.7375 - val_accuracy: 0.5706 - val_loss: 1.0638
Epoch 8/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 51s 526ms/step - accuracy: 0.7397 - loss: 0.6897 - val_accuracy: 0.5506 - val_loss: 1.1313
Epoch 9/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 483ms/step - accuracy: 0.7336 - loss: 0.6846

96/96 ━━━━━━━━━━━━━━━━━━━━ 50s 519ms/step - accuracy: 0.7335 - loss: 0.6849 - val_accuracy: 0.5736 - val_loss: 1.0640
Epoch 10/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 506ms/step - accuracy: 0.7468 - loss: 0.6444

96/96 ━━━━━━━━━━━━━━━━━━━━ 52s 544ms/step - accuracy: 0.7467 - loss: 0.6445 - val_accuracy: 0.5874 - val_loss: 1.1013


In [73]:
base_model.trainable = True
for layer in base_model.layers[:-30]:  # freeze first layers
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5),  # low LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_stage2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=[early_stop, checkpoint]
)


Epoch 1/5
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 570ms/step - accuracy: 0.5614 - loss: 1.2160

96/96 ━━━━━━━━━━━━━━━━━━━━ 85s 701ms/step - accuracy: 0.5617 - loss: 1.2148 - val_accuracy: 0.5920 - val_loss: 1.1109
Epoch 2/5
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 513ms/step - accuracy: 0.6642 - loss: 0.8538

96/96 ━━━━━━━━━━━━━━━━━━━━ 53s 551ms/step - accuracy: 0.6642 - loss: 0.8536 - val_accuracy: 0.5936 - val_loss: 1.1308
Epoch 3/5
96/96 ━━━━━━━━━━━━━━━━━━━━ 52s 541ms/step - accuracy: 0.6909 - loss: 0.8140 - val_accuracy: 0.5890 - val_loss: 1.1320
Epoch 4/5
96/96 ━━━━━━━━━━━━━━━━━━━━ 51s 531ms/step - accuracy: 0.7110 - loss: 0.7418 - val_accuracy: 0.5874 - val_loss: 1.1201
Epoch 5/5
96/96 ━━━━━━━━━━━━━━━━━━━━ 52s 541ms/step - accuracy: 0.7148 - loss: 0.7216 - val_accuracy: 0.5874 - val_loss: 1.1157


In [75]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc*100:.2f}%")

21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 141ms/step - accuracy: 0.5670 - loss: 1.1641
Test Accuracy: 62.06%


In [76]:
model.save("era_classifier_stage1.h5")


In [77]:
# Path to save in your Drive
model_save_path = '/content/drive/MyDrive/eras_models/era_classifier_stage1.h5'

# Save the model
model.save(model_save_path)
print(f"Model saved to: {model_save_path}")


Model saved to: /content/drive/MyDrive/eras_models/era_classifier_stage1.h5


In [78]:
# This will show the mapping from class name to index
print(train_generator.class_indices)


{'Coptic egypt': 0, 'Greco_Roman': 1, 'Islamic ': 2, 'Ottoman': 3, 'Pharaonic': 4}
